In [ ]:
import os
import random
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ────────────────────────────────────────────────
# 1. Load Supabase credentials
# ────────────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ────────────────────────────────────────────────
# 2. Load data
# ────────────────────────────────────────────────
data = pd.read_csv("/Users/othmanbensouda/Desktop/debt_collection_website/files/get_plaintiffs_defendants.csv")

# ────────────────────────────────────────────────
# 3. Assign annotators randomly (Brian, Parker, Victor)
# ────────────────────────────────────────────────
cases = list(data["CASE_NUMBER"].unique())
random.seed(42)
random.shuffle(cases)
n = len(cases)

part1, part2, part3 = cases[: n // 3], cases[n // 3 : 2 * n // 3], cases[2 * n // 3 :]

def assign_annotator(case_number):
    if case_number in part1:
        return "Brian"
    elif case_number in part2:
        return "Parker"
    else:
        return "Victor"

data["annotator_id"] = data["CASE_NUMBER"].apply(assign_annotator)

# ────────────────────────────────────────────────
# 4. Clean up text fields (avoid NaN/float errors)
# ────────────────────────────────────────────────
data["FULL_NAME"] = data["FULL_NAME"].fillna("").astype(str)
data["PERSON_ROLE"] = data["PERSON_ROLE"].fillna("").astype(str)

# ────────────────────────────────────────────────
# 5. Aggregate one row per case
# ────────────────────────────────────────────────
grouped = (
    data.groupby("CASE_NUMBER")
    .apply(lambda g: pd.Series({
        "case_number": g.name,
        "case_hashkey": g["CASE_HASHKEY"].iloc[0] if "CASE_HASHKEY" in g else None,
        "complaint_filed_date": g["COMPLAINT_FILED_DATE"].iloc[0] if "COMPLAINT_FILED_DATE" in g else None,
        "plaintiff": [str(x) for x in g.loc[g["PERSON_ROLE"].str.lower() == "plaintiff", "FULL_NAME"] if pd.notna(x) and x],
        "defendant": [str(x) for x in g.loc[g["PERSON_ROLE"].str.lower() == "defendant", "FULL_NAME"] if pd.notna(x) and x],
        "annotator_id": g["annotator_id"].iloc[0],
    }))
    .reset_index(drop=True)
)

# ────────────────────────────────────────────────
# 6. Upload to Supabase (batch, fast)
# ────────────────────────────────────────────────
records = grouped.to_dict(orient="records")

response = supabase.table("cases_gold").upsert(records).execute()

print("✅ Upload complete!")
print("Inserted/updated rows:", len(records))
print("\nAnnotator distribution:")
print(grouped["annotator_id"].value_counts())

# Optional: preview one example
print("\nExample row:")
print(grouped.head(1).to_dict(orient="records")[0])


/var/folders/k7/b0_b7t6j6n72t68sh4s7t8400000gn/T/ipykernel_28487/84020101.py:51: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


✅ Upload complete!
Inserted/updated rows: 5

Annotator distribution:
annotator_id
Victor    2
Parker    2
Brian     1
Name: count, dtype: int64

Example row:
{'case_number': '23CHLC16737', 'case_hashkey': 221885112732742272, 'complaint_filed_date': '2023-06-29 00:00:00.000', 'plaintiff': ['DNF Associates, LLC'], 'defendant': ['SPENCY ARINGO'], 'annotator_id': 'Victor'}


In [ ]:
import openpyxl
grouped.to_excel("/Users/othmanbensouda/Desktop/debt_collection_website/files/cases_assigned.xlsx")

# Round 2 (inter-annotator agreement)

In [3]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ───────────────────────────────────────────
# 1. Load Supabase credentials
# ───────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ───────────────────────────────────────────
# 2. Fetch ROUND 1 cases
# ───────────────────────────────────────────
res = (
    supabase.table("cases_gold")
    .select("*")
    .eq("round", 1)
    .execute()
)

raw_df = pd.DataFrame(res.data)
print("Loaded round 1 cases total:", len(raw_df))

# Keep ONLY the 3 annotators involved in IAA
df = raw_df[raw_df["annotator_id"].isin(["Parker", "Brian", "Victor"])].copy()

print("Cases retained for IAA:", len(df))
print(df["annotator_id"].value_counts())

# Remove accidental duplicates
df = df.drop_duplicates(subset=["case_number"])

# ───────────────────────────────────────────
# 3. Stable perfect split (deterministic)
# ───────────────────────────────────────────
def split_half(lst):
    lst = sorted(lst)
    n = len(lst)
    half = n // 2
    return lst[:half], lst[half:]

# Split by annotator
parker_cases = df[df.annotator_id == "Parker"]["case_number"].tolist()
brian_cases  = df[df.annotator_id == "Brian"]["case_number"].tolist()
victor_cases = df[df.annotator_id == "Victor"]["case_number"].tolist()

parker1, parker2 = split_half(parker_cases)
brian1,  brian2  = split_half(brian_cases)
victor1, victor2 = split_half(victor_cases)

# ───────────────────────────────────────────
# 4. Create ROUND 2 assignments
# ───────────────────────────────────────────
round2 = []

# Parker → Brian + Victor
round2 += [{"case_number": c, "annotator_id": "Brian"} for c in parker1]
round2 += [{"case_number": c, "annotator_id": "Victor"} for c in parker2]

# Brian → Parker + Victor
round2 += [{"case_number": c, "annotator_id": "Parker"} for c in brian1]
round2 += [{"case_number": c, "annotator_id": "Victor"} for c in brian2]

# Victor → Brian + Parker
round2 += [{"case_number": c, "annotator_id": "Brian"} for c in victor1]
round2 += [{"case_number": c, "annotator_id": "Parker"} for c in victor2]

round2_df = pd.DataFrame(round2)

# ───────────────────────────────────────────
# 5. VERIFICATIONS BEFORE INSERTING
# ───────────────────────────────────────────
print("\n🔍 Verifying…")

n1 = df["case_number"].nunique()
n2 = round2_df["case_number"].nunique()
print("Round1 unique (3 annotators only):", n1)
print("Round2 unique:", n2)

assert n1 == n2, "❌ Round 2 does NOT contain the same unique cases!"

# No duplicates
assert round2_df["case_number"].duplicated().sum() == 0, "❌ Duplicate cases in round 2!"

# No self-assign
merged = round2_df.merge(df[["case_number", "annotator_id"]], on="case_number", suffixes=("_r2", "_r1"))
assert (merged.annotator_id_r1 == merged.annotator_id_r2).sum() == 0, "❌ Self-assignment detected!"

# Distribution OK
print(round2_df["annotator_id"].value_counts())

# Set equality
assert set(df["case_number"]) == set(round2_df["case_number"]), "❌ Case sets differ!"

# ───────────────────────────────────────────
# 6. Add metadata + insert
# ───────────────────────────────────────────
final = round2_df.merge(
    df[["case_number", "plaintiff", "defendant", "complaint_filed_date", "case_hashkey"]],
    on="case_number",
    how="left"
)

final["round"] = 2
final["progress"] = "incomplete"

records = final.to_dict(orient="records")

print("\n🚀 Inserting round 2 into Supabase…")
supabase.table("cases_gold").insert(records).execute()

print("✅ ROUND 2 SUCCESSFULLY INSERTED!")
print("Inserted rows:", len(records))
print(final["annotator_id"].value_counts())


Loaded round 1 cases total: 111
Cases retained for IAA: 105
annotator_id
Parker    35
Victor    35
Brian     35
Name: count, dtype: int64

🔍 Verifying…
Round1 unique (3 annotators only): 105
Round2 unique: 105
annotator_id
Victor    36
Parker    35
Brian     34
Name: count, dtype: int64

🚀 Inserting round 2 into Supabase…
✅ ROUND 2 SUCCESSFULLY INSERTED!
Inserted rows: 105
annotator_id
Victor    36
Parker    35
Brian     34
Name: count, dtype: int64


In [7]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ──────────────────────────────────────────────
# 1. Load Supabase
# ──────────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

ANNOTATORS = ["Parker", "Brian", "Victor"]
IGNORE = {"time_caselevel", "created_at"}

# ──────────────────────────────────────────────
# Function to compare for one annotator
# ──────────────────────────────────────────────
def compare_for(annotator):
    print("\n" + "="*60)
    print(f"🔎 Checking annotator: {annotator}")
    print("="*60)

    # Fetch gold
    df_gold = pd.DataFrame(
        supabase.table("results_gold")
        .select("*")
        .eq("annotator_id", annotator)
        .execute().data
    )

    # Fetch fallback
    df_fb = pd.DataFrame(
        supabase.table("results_gold_fallback")
        .select("*")
        .eq("annotator_id", annotator)
        .execute().data
    )

    print(f"rows in results_gold: {len(df_gold)}")
    print(f"rows in fallback: {len(df_fb)}")

    if df_gold.empty or df_fb.empty:
        print(f"❌ No data for annotator {annotator}")
        return

    # Keep last version per case
    last_gold = (
        df_gold.sort_values(["case_number", "version"])
        .groupby("case_number")
        .tail(1)
        .reset_index(drop=True)
    )
    last_fb = (
        df_fb.sort_values(["case_number", "version"])
        .groupby("case_number")
        .tail(1)
        .reset_index(drop=True)
    )

    cases_gold = set(last_gold.case_number)
    cases_fb = set(last_fb.case_number)

    common = sorted(cases_gold & cases_fb)

    # Warn if mismatch
    if cases_gold != cases_fb:
        print("⚠️ Case mismatch:")
        print("  In gold only:", cases_gold - cases_fb)
        print("  In fallback only:", cases_fb - cases_gold)

    # Compare ignoring IGNORE fields
    diffs = []
    for case in common:
        g = last_gold[last_gold.case_number == case].iloc[0].to_dict()
        f = last_fb[last_fb.case_number == case].iloc[0].to_dict()

        cols = set(g.keys()) | set(f.keys())
        for col in cols:
            if col in IGNORE:
                continue

            gv = g.get(col)
            fv = f.get(col)

            if isinstance(gv, list): gv = sorted(gv)
            if isinstance(fv, list): fv = sorted(fv)

            if pd.isna(gv) and pd.isna(fv):
                continue

            if gv != fv:
                diffs.append({
                    "case_number": case,
                    "column": col,
                    "gold": gv,
                    "fallback": fv
                })

    if not diffs:
        print(f"✅ PERFECT MATCH for {annotator} (ignoring time_caselevel + created_at)")
    else:
        print(f"❌ DIFFERENCES FOUND for {annotator}:")
        print(pd.DataFrame(diffs).to_string(index=False))


# ──────────────────────────────────────────────
# Run comparison for all three
# ──────────────────────────────────────────────
for annotator in ANNOTATORS:
    compare_for(annotator)





🔎 Checking annotator: Parker
rows in results_gold: 108
rows in fallback: 71
⚠️ Case mismatch:
  In gold only: {'23CHLC16737'}
  In fallback only: set()
✅ PERFECT MATCH for Parker (ignoring time_caselevel + created_at)

🔎 Checking annotator: Brian
rows in results_gold: 125
rows in fallback: 90
✅ PERFECT MATCH for Brian (ignoring time_caselevel + created_at)

🔎 Checking annotator: Victor
rows in results_gold: 95
rows in fallback: 60
✅ PERFECT MATCH for Victor (ignoring time_caselevel + created_at)


# Upload second batch (100 initial cases in the excel sheet)

In [2]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ────────────────────────────────────────────────
# 1. Load Supabase credentials
# ────────────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ────────────────────────────────────────────────
# 2. Load data from the specified CSV
# ────────────────────────────────────────────────
data = pd.read_csv("/Users/othmanbensouda/Desktop/debt_collection_website/files/get_plaintiffs_defendants_initial_100.csv")

# ────────────────────────────────────────────────
# 3. Clean up text fields (avoid NaN/float errors)
# ────────────────────────────────────────────────
data["FULL_NAME"] = data["FULL_NAME"].fillna("").astype(str)
data["PERSON_ROLE"] = data["PERSON_ROLE"].fillna("").astype(str)

# ────────────────────────────────────────────────
# 4. Aggregate one row per case
# ────────────────────────────────────────────────
grouped = (
    data.groupby("CASE_NUMBER")
    .apply(lambda g: pd.Series({
        "case_number": g.name,
        "case_hashkey": g["CASE_HASHKEY"].iloc[0] if "CASE_HASHKEY" in g else None,
        "complaint_filed_date": g["COMPLAINT_FILED_DATE"].iloc[0] if "COMPLAINT_FILED_DATE" in g else None,
        "plaintiff": [str(x) for x in g.loc[g["PERSON_ROLE"].str.lower() == "plaintiff", "FULL_NAME"] if pd.notna(x) and x],
        "defendant": [str(x) for x in g.loc[g["PERSON_ROLE"].str.lower() == "defendant", "FULL_NAME"] if pd.notna(x) and x],
        "annotator_id": None,  # No annotator assigned for batch 2
        "batch": 2  # Set batch to 2
    }))
    .reset_index(drop=True)
)

# ────────────────────────────────────────────────
# 5. Upload to Supabase (batch, fast)
# ────────────────────────────────────────────────
records = grouped.to_dict(orient="records")

response = supabase.table("cases_gold").upsert(records).execute()

print("✅ Upload complete!")
print("Inserted/updated rows:", len(records))
print("All rows set to batch = 2")

# Optional: preview one example
print("\nExample row:")
print(grouped.head(1).to_dict(orient="records")[0])

/var/folders/k7/b0_b7t6j6n72t68sh4s7t8400000gn/T/ipykernel_21564/1567906470.py:30: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


✅ Upload complete!
Inserted/updated rows: 99
All rows set to batch = 2

Example row:
{'case_number': '23CHLC04088', 'case_hashkey': 938445792436271488, 'complaint_filed_date': '2023-02-14 00:00:00.000', 'plaintiff': ['INVESTMENT RETRIEVERS, INC.'], 'defendant': ['KANDIS MCDONALD'], 'annotator_id': None, 'batch': 2}


# Send initial 100 debt collection cases (the ones in the excel sheet) google drive links to supabase

In [3]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ────────────────────────────────────────────────
# 1. Load Supabase credentials
# ────────────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ────────────────────────────────────────────────
# 2. Load data from CSV
# ────────────────────────────────────────────────
data = pd.read_csv("/Users/othmanbensouda/Desktop/debt_collection_website/files/initial_100_debt_collection_gold_list.csv")

print("Column names in CSV:")
print(data.columns.tolist())
print("\nFirst few rows:")
print(data.head())

# ────────────────────────────────────────────────
# 3. Clean up and prepare data
# ────────────────────────────────────────────────
# Fill NaN values with empty strings to avoid issues
data = data.fillna("")

# Convert to records for upload
records = data.to_dict(orient="records")

# ────────────────────────────────────────────────
# 4. Upload to Supabase gdrive_files table
# ────────────────────────────────────────────────
response = supabase.table("gdrive_files").upsert(records).execute()

print("\n✅ Upload complete!")
print("Inserted/updated rows:", len(records))

Column names in CSV:
['CASE_NUMBER', 'DOCUMENT_ID', 'DOCUMENT_FILED_DATE', 'DOCUMENT_NAME', 'DOCUMENT_NAME_DRIVE', 'LINK_DRIVE']

First few rows:
   CASE_NUMBER  DOCUMENT_ID      DOCUMENT_FILED_DATE  \
0  23CHLC04088     91598076  2023-02-14 00:00:00.000   
1  23CHLC04088     91598077  2023-02-14 00:00:00.000   
2  23CHLC04088     91598078  2023-02-14 00:00:00.000   
3  23CHLC04088     91598079  2023-02-14 00:00:00.000   
4  23CHLC04088     91598080  2023-02-14 00:00:00.000   

                                       DOCUMENT_NAME  \
0                                          Complaint   
1                                            Summons   
2                       Declaration (name extension)   
3                             Civil Case Cover Sheet   
4  Order to Show Cause Hearing/Case Management Re...   

                DOCUMENT_NAME_DRIVE  \
0    complaint_23chlc04088_91598076   
1                               NaN   
2  declaration_23chlc04088_91598078   
3                       

# Assign the initial 100 cases randomly (the ones in the excel sheet)

In [6]:
import os
import random
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ────────────────────────────────────────────────
# 1. Load Supabase credentials
# ────────────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ────────────────────────────────────────────────
# 2. Fetch all batch 2 cases from cases_gold
# ────────────────────────────────────────────────
response = supabase.table("cases_gold").select("*").eq("batch", 2).execute()
batch_2_cases = response.data

if not batch_2_cases:
    print("⚠️ No cases found in batch 2")
    exit()

print(f"Found {len(batch_2_cases)} cases in batch 2")

# ────────────────────────────────────────────────
# 3. Assign annotators randomly (Brian, Parker, Victor)
# ────────────────────────────────────────────────
case_numbers = [case["case_number"] for case in batch_2_cases]
random.seed(42)
random.shuffle(case_numbers)
n = len(case_numbers)

part1 = case_numbers[: n // 3]
part2 = case_numbers[n // 3 : 2 * n // 3]
part3 = case_numbers[2 * n // 3 :]

def assign_annotator(case_number):
    if case_number in part1:
        return "Brian"
    elif case_number in part2:
        return "Parker"
    else:
        return "Victor"

# ────────────────────────────────────────────────
# 4. Update each case with annotator_id
# ────────────────────────────────────────────────
updates = []
successful_updates = 0
failed_updates = 0

for case in batch_2_cases:
    annotator = assign_annotator(case["case_number"])
    
    # Update in Supabase
    try:
        response = supabase.table("cases_gold")\
            .update({"annotator_id": annotator})\
            .eq("case_number", case["case_number"])\
            .eq("batch", 2)\
            .execute()
        successful_updates += 1
    except Exception as e:
        print(f"Error updating case {case['case_number']}: {e}")
        failed_updates += 1
    
    # Store for Excel export
    updates.append({
        "case_number": case["case_number"],
        "batch": 2,
        "case_hashkey": case.get("case_hashkey"),
        "complaint_filed_date": case.get("complaint_filed_date"),
        "plaintiff": case.get("plaintiff"),
        "defendant": case.get("defendant"),
        "annotator_id": annotator
    })

print("✅ Assignment complete!")
print(f"Successfully updated: {successful_updates}")
print(f"Failed updates: {failed_updates}")
print(f"Total cases: {len(batch_2_cases)}")

# ────────────────────────────────────────────────
# 5. Create DataFrame and save to Excel
# ────────────────────────────────────────────────
df = pd.DataFrame(updates)

# Sort by annotator for easier review
df = df.sort_values(["annotator_id", "case_number"]).reset_index(drop=True)

# Save to Excel
output_path = "/Users/othmanbensouda/Desktop/debt_collection_website/files/batch_2_case_assignments.xlsx"
df.to_excel(output_path, index=False, engine='openpyxl')

print(f"\n✅ Excel file saved to: {output_path}")

# Show distribution
print("\nAnnotator distribution:")
print(df["annotator_id"].value_counts().sort_index())

# Show sample of each annotator's assignments
print("\nSample assignments per annotator:")
for annotator in ["Brian", "Parker", "Victor"]:
    print(f"\n{annotator}:")
    annotated_cases = df[df["annotator_id"] == annotator]["case_number"].head(3).tolist()
    print(annotated_cases)

Found 99 cases in batch 2
✅ Assignment complete!
Successfully updated: 99
Failed updates: 0
Total cases: 99

✅ Excel file saved to: /Users/othmanbensouda/Desktop/debt_collection_website/files/batch_2_case_assignments.xlsx

Annotator distribution:
annotator_id
Brian     33
Parker    33
Victor    33
Name: count, dtype: int64

Sample assignments per annotator:

Brian:
['23CHLC12580', '23NWLC25562', '23NWLC29036']

Parker:
['23CHLC12118', '23CHLC16737', '23CHLC22869']

Victor:
['23CHLC04088', '23CHLC18504', '23CHLC18998']


# Shuffle cases and assign batch 2, round 2 (initial 100 cases in excel sheet) 

In [3]:
import os
import random
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ────────────────────────────────────────────────
# 1. Load existing assignments from Excel
# ────────────────────────────────────────────────
excel_path = "/Users/othmanbensouda/Desktop/debt_collection_website/files/batch_2_case_assignments.xlsx"
df_old = pd.read_excel(excel_path)

print("📊 Previous assignments (Round 1):")
print("\nDistribution:")
print(df_old["annotator_id"].value_counts().sort_index())

# ────────────────────────────────────────────────
# 2. Load Supabase credentials
# ────────────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ────────────────────────────────────────────────
# 3. Fetch all batch 2 cases from cases_gold
# ────────────────────────────────────────────────
response = supabase.table("cases_gold").select("*").eq("batch", 2).execute()
batch_2_cases = response.data

if not batch_2_cases:
    print("⚠️ No cases found in batch 2")
    exit()

print(f"\n🔍 Found {len(batch_2_cases)} cases in batch 2")

# ────────────────────────────────────────────────
# 4. Create mapping of old assignments
# ────────────────────────────────────────────────
old_assignments = df_old.set_index("case_number")["annotator_id"].to_dict()

# ────────────────────────────────────────────────
# 5. Shuffle to give different cases (TRUE random shuffle with EQUAL distribution)
# ────────────────────────────────────────────────
# Get all case numbers
case_numbers = [case["case_number"] for case in batch_2_cases]

# Separate by old annotator
brian_old = set([cn for cn in case_numbers if old_assignments.get(cn) == "Brian"])
parker_old = set([cn for cn in case_numbers if old_assignments.get(cn) == "Parker"])
victor_old = set([cn for cn in case_numbers if old_assignments.get(cn) == "Victor"])

print(f"\n📋 Round 1 assignments:")
print(f"  Brian: {len(brian_old)} cases")
print(f"  Parker: {len(parker_old)} cases")
print(f"  Victor: {len(victor_old)} cases")

# Calculate exactly how many cases each person should get
total_cases = len(case_numbers)
cases_per_person = total_cases // 3
remainder = total_cases % 3

# Each person gets cases_per_person, and the first 'remainder' people get 1 extra
brian_target = cases_per_person + (1 if remainder > 0 else 0)
parker_target = cases_per_person + (1 if remainder > 1 else 0)
victor_target = cases_per_person

print(f"\n🎯 Target distribution:")
print(f"  Brian: {brian_target} cases")
print(f"  Parker: {parker_target} cases")
print(f"  Victor: {victor_target} cases")

# Shuffle all cases randomly
random.seed(42)  # For reproducibility
shuffled_cases = case_numbers.copy()
random.shuffle(shuffled_cases)

# Assign cases ensuring no one gets their own cases
new_assignments = {}
brian_cases = []
parker_cases = []
victor_cases = []

for case_num in shuffled_cases:
    # Try to assign to someone who didn't review it before
    if case_num not in brian_old and len(brian_cases) < brian_target:
        brian_cases.append(case_num)
        new_assignments[case_num] = "Brian"
    elif case_num not in parker_old and len(parker_cases) < parker_target:
        parker_cases.append(case_num)
        new_assignments[case_num] = "Parker"
    elif case_num not in victor_old and len(victor_cases) < victor_target:
        victor_cases.append(case_num)
        new_assignments[case_num] = "Victor"
    # If all preferred options are full, assign to whoever has space
    elif len(brian_cases) < brian_target:
        brian_cases.append(case_num)
        new_assignments[case_num] = "Brian"
    elif len(parker_cases) < parker_target:
        parker_cases.append(case_num)
        new_assignments[case_num] = "Parker"
    elif len(victor_cases) < victor_target:
        victor_cases.append(case_num)
        new_assignments[case_num] = "Victor"

print(f"\n🔄 Round 2 assignments (randomly shuffled with equal distribution):")
print(f"  Brian: {len(brian_cases)} cases")
print(f"  Parker: {len(parker_cases)} cases")
print(f"  Victor: {len(victor_cases)} cases")

# ────────────────────────────────────────────────
# 6. INSERT new rows in Supabase for round 2 (DO NOT UPDATE round 1!)
# ────────────────────────────────────────────────
print("\n📤 Creating NEW rows in Supabase for Round 2...")
updates = []
successful_inserts = 0
failed_inserts = 0

for case in batch_2_cases:
    case_num = case["case_number"]
    new_annotator = new_assignments[case_num]
    
    # CREATE NEW ROW in Supabase for round 2 (keep round 1 intact!)
    try:
        # Copy all fields from the round 1 case
        new_row = {
            "case_number": case_num,
            "batch": 2,
            "round": 2,  # NEW ROUND
            "annotator_id": new_annotator,  # NEW ANNOTATOR
            "case_hashkey": case.get("case_hashkey"),
            "complaint_filed_date": case.get("complaint_filed_date"),
            "plaintiff": case.get("plaintiff"),
            "defendant": case.get("defendant"),
            # Copy any other fields that exist
            "case_title": case.get("case_title"),
            "filing_date": case.get("filing_date"),
            "close_date": case.get("close_date"),
            "disposition": case.get("disposition"),
            "case_type": case.get("case_type"),
        }
        
        # Remove None values
        new_row = {k: v for k, v in new_row.items() if v is not None}
        
        response = supabase.table("cases_gold").insert(new_row).execute()
        successful_inserts += 1
    except Exception as e:
        print(f"❌ Error inserting case {case_num}: {e}")
        failed_inserts += 1
    
    # Store for Excel export
    updates.append({
        "case_number": case_num,
        "batch": 2,
        "round": 2,
        "case_hashkey": case.get("case_hashkey"),
        "complaint_filed_date": case.get("complaint_filed_date"),
        "plaintiff": case.get("plaintiff"),
        "defendant": case.get("defendant"),
        "annotator_id": new_annotator,
        "previous_annotator_round_1": old_assignments.get(case_num, "Unknown")
    })

print(f"\n✅ Supabase insert complete!")
print(f"  Successfully inserted: {successful_inserts}")
print(f"  Failed inserts: {failed_inserts}")
print(f"  Total new rows: {len(batch_2_cases)}")

# ────────────────────────────────────────────────
# 7. Create DataFrame and save to Excel
# ────────────────────────────────────────────────
df_new = pd.DataFrame(updates)
df_new = df_new.sort_values(["annotator_id", "case_number"]).reset_index(drop=True)

output_path = "/Users/othmanbensouda/Desktop/debt_collection_website/files/batch_2_round_2_case_assignments.xlsx"
df_new.to_excel(output_path, index=False, engine='openpyxl')

print(f"\n✅ Excel file saved to: {output_path}")

# Show distribution
print("\n📊 Round 2 final distribution:")
print(df_new["annotator_id"].value_counts().sort_index())

# Verify no one has the same cases
print("\n🔍 Verification - checking if anyone got the same cases:")
same_cases = df_new[df_new["annotator_id"] == df_new["previous_annotator_round_1"]]
print(f"  Cases where annotator stayed the same: {len(same_cases)} (should be 0)")
if len(same_cases) > 0:
    print("  ⚠️ WARNING: Some annotators got the same cases!")
    print(same_cases[["case_number", "annotator_id", "previous_annotator_round_1"]])
else:
    print("  ✅ Perfect! Everyone has completely different cases for Round 2.")

print("\n🎉 All done! The assignments have been:")
print("  1. Rotated so everyone reviews different cases")
print("  2. Updated in Supabase (batch=2, round=2)")
print("  3. Saved to new Excel file")

📊 Previous assignments (Round 1):

Distribution:
annotator_id
Brian     33
Parker    33
Victor    33
Name: count, dtype: int64

🔍 Found 99 cases in batch 2

📋 Round 1 assignments:
  Brian: 33 cases
  Parker: 33 cases
  Victor: 33 cases

🎯 Target distribution:
  Brian: 33 cases
  Parker: 33 cases
  Victor: 33 cases

🔄 Round 2 assignments (randomly shuffled with equal distribution):
  Brian: 33 cases
  Parker: 33 cases
  Victor: 33 cases

📤 Creating NEW rows in Supabase for Round 2...

✅ Supabase insert complete!
  Successfully inserted: 99
  Failed inserts: 0
  Total new rows: 99

✅ Excel file saved to: /Users/othmanbensouda/Desktop/debt_collection_website/files/batch_2_round_2_case_assignments.xlsx

📊 Round 2 final distribution:
annotator_id
Brian     33
Parker    33
Victor    33
Name: count, dtype: int64

🔍 Verification - checking if anyone got the same cases:
  Cases where annotator stayed the same: 7 (should be 0)
  ⚠️ WARNING: Some annotators got the same cases!
    case_number ann

# Batch 3 round 1

## Send cases to google drive and create file with drive links

In [17]:
import pandas as pd
import os
import random
import re
from tqdm import tqdm
from io import StringIO

# Google Drive OAuth
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.transport.requests import Request
from googleapiclient.http import MediaFileUpload
import pickle

# =====================================================
# CONFIG
# =====================================================

CSV_PATH = "/Users/othmanbensouda/Desktop/debt_collection_website/files/rejections_cases.csv"
PDF_FOLDER = "/Users/othmanbensouda/Desktop/debt_collection_website/files/rejections"
OUTPUT_CSV = "20_rejections_with_drive_links.csv"

SCOPES = ['https://www.googleapis.com/auth/drive']

# =====================================================
# STEP 1 — GET CASES FROM ACTUAL PDF FILES
# =====================================================

pdf_files = [f for f in os.listdir(PDF_FOLDER) if f.endswith(".pdf")]

records = []
for f in pdf_files:
    case_number = f.split("_")[0]
    document_id = f.split("_")[1].replace(".pdf", "")
    records.append((case_number, document_id))

folder_df = pd.DataFrame(records, columns=["CASE_NUMBER", "DOCUMENT_ID"])

unique_cases = folder_df["CASE_NUMBER"].unique().tolist()
print(f"Found {len(unique_cases)} cases in folder")

if len(unique_cases) < 20:
    raise ValueError("Less than 20 cases available in folder.")

random_cases = random.sample(unique_cases, 20)
folder_df = folder_df[folder_df["CASE_NUMBER"].isin(random_cases)]

print(f"Selected {len(random_cases)} cases")

# =====================================================
# STEP 2 — LOAD AND FIX CSV
# =====================================================

with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
    lines = f.readlines()

header = lines[0].strip()
cleaned_lines = [header]

for line in lines[1:]:
    line = line.strip()
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1]
    line = line.replace('""', '"')
    cleaned_lines.append(line)

cleaned_csv = StringIO("\n".join(cleaned_lines))
df = pd.read_csv(cleaned_csv)

df = df[[
    "CASE_NUMBER",
    "DOCUMENT_ID",
    "DOCUMENT_FILED_DATE",
    "DOCUMENT_NAME"
]].copy()

df["CASE_NUMBER"] = df["CASE_NUMBER"].astype(str).str.strip()
df["DOCUMENT_ID"] = (
    df["DOCUMENT_ID"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

# Keep only files that physically exist
df_sample = pd.merge(
    folder_df,
    df,
    on=["CASE_NUMBER", "DOCUMENT_ID"],
    how="left"
)

df_sample["LINK_DRIVE"] = None

print(f"Uploading {len(df_sample)} actual PDFs")

# =====================================================
# STEP 3 — GOOGLE DRIVE OAUTH AUTHENTICATION
# =====================================================

creds = None

if os.path.exists('token.pickle'):
    with open('token.pickle', 'rb') as token:
        creds = pickle.load(token)

if not creds or not creds.valid:
    flow = InstalledAppFlow.from_client_secrets_file(
        'credentials.json',
        SCOPES
    )
    creds = flow.run_local_server(port=0)

    with open('token.pickle', 'wb') as token:
        pickle.dump(creds, token)

drive_service = build('drive', 'v3', credentials=creds)

# =====================================================
# STEP 4 — CREATE PARENT FOLDER
# =====================================================

parent_folder = drive_service.files().create(
    body={
        "name": "debt_collection_sample_20",
        "mimeType": "application/vnd.google-apps.folder"
    },
    fields="id"
).execute()

parent_folder_id = parent_folder["id"]

print("Created parent folder in your Drive")

# =====================================================
# FILE NAMING
# =====================================================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", "_", text.strip())
    return text


def generate_filename(case_number, document_id, document_name):
    name_lower = str(document_name).lower()

    if "request" in name_lower and "default" in name_lower:
        prefix = "request_for_default_judgment"
    elif "declaration" in name_lower:
        prefix = "declaration"
    elif "complaint" in name_lower:
        prefix = "complaint"
    elif "rejection" in name_lower:
        prefix = "rejection"
    else:
        prefix = clean_text(document_name)

    return f"{prefix}_{case_number.lower()}_{document_id}.pdf"

# =====================================================
# STEP 5 — UPLOAD FILES
# =====================================================

for case in tqdm(random_cases, desc="Uploading cases"):

    case_rows = df_sample[df_sample["CASE_NUMBER"] == case]

    folder = drive_service.files().create(
        body={
            "name": case,
            "mimeType": "application/vnd.google-apps.folder",
            "parents": [parent_folder_id]
        },
        fields="id"
    ).execute()

    folder_id = folder["id"]

    for idx, row in tqdm(
        case_rows.iterrows(),
        total=len(case_rows),
        desc=f"{case}",
        leave=False
    ):

        case_number = row["CASE_NUMBER"]
        document_id = row["DOCUMENT_ID"]
        document_name = row["DOCUMENT_NAME"]

        local_pdf_path = os.path.join(
            PDF_FOLDER,
            f"{case_number}_{document_id}.pdf"
        )

        new_filename = generate_filename(case_number, document_id, document_name)

        uploaded = drive_service.files().create(
            body={
                "name": new_filename,
                "parents": [folder_id]
            },
            media_body=MediaFileUpload(local_pdf_path, mimetype="application/pdf"),
            fields="id, webViewLink"
        ).execute()

        df_sample.loc[idx, "LINK_DRIVE"] = uploaded["webViewLink"]

# =====================================================
# STEP 6 — SAVE OUTPUT
# =====================================================

df_sample.to_csv(OUTPUT_CSV, index=False)

print("✅ DONE — Files uploaded and CSV saved.")

Found 21 cases in folder
Selected 20 cases
Uploading 424 actual PDFs
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=513638550177-mnh58osn0obnss0i0dmjvktrn0ftoc6l.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A50880%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&state=wlTj8l2Jwajpc8HQne75UZajYwnSFg&access_type=offline
Created parent folder in your Drive


Uploading cases: 100%|██████████| 20/20 [12:35<00:00, 37.75s/it]

✅ DONE — Files uploaded and CSV saved.


## Send info to gdrive files database on supabase

In [3]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ────────────────────────────────────────────────
# 1. Load Supabase credentials
# ────────────────────────────────────────────────
load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("Supabase credentials not found in .env file.")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ────────────────────────────────────────────────
# 2. Load the 20 rejections CSV
# ────────────────────────────────────────────────
CSV_PATH = "20_rejections_with_drive_links.csv"

data = pd.read_csv(CSV_PATH)

print("Column names in CSV:")
print(data.columns.tolist())
print("\nFirst few rows:")
print(data.head())

# ────────────────────────────────────────────────
# 3. Clean & Prepare Data
# ────────────────────────────────────────────────

# Fill NaNs
data = data.fillna("")

# Ensure strings (important for Supabase)
data["CASE_NUMBER"] = data["CASE_NUMBER"].astype(str)
data["DOCUMENT_ID"] = data["DOCUMENT_ID"].astype(str)
data["DOCUMENT_FILED_DATE"] = data["DOCUMENT_FILED_DATE"].astype(str)
data["DOCUMENT_NAME"] = data["DOCUMENT_NAME"].astype(str)
data["LINK_DRIVE"] = data["LINK_DRIVE"].astype(str)

records = data.to_dict(orient="records")

print(f"\nUploading {len(records)} rows to Supabase...")

# ────────────────────────────────────────────────
# 4. Upload to Supabase
# ────────────────────────────────────────────────

response = supabase.table("gdrive_files").upsert(records).execute()

print("\n✅ Upload complete!")
print("Inserted/updated rows:", len(records))

Column names in CSV:
['CASE_NUMBER', 'DOCUMENT_ID', 'DOCUMENT_FILED_DATE', 'DOCUMENT_NAME', 'LINK_DRIVE']

First few rows:
   CASE_NUMBER  DOCUMENT_ID      DOCUMENT_FILED_DATE  \
0  24CHLC00819     98987822  2024-03-01 00:00:00.000   
1  24CHLC00911     97924155  2024-01-10 00:00:00.000   
2  24CHLC00396     99753713  2024-04-09 00:00:00.000   
3  24CHLC01576    113252449  2025-11-18 00:00:00.000   
4  24CHLC01076     98090515  2024-01-19 00:00:00.000   

                                       DOCUMENT_NAME  \
0                    Declaration of Costs - CCP 1033   
1  Order to Show Cause Hearing/Case Management Re...   
2      Abstract of Judgment - Civil and Small Claims   
3                       Declaration (name extension)   
4            Proof of Service by Substituted Service   

                                          LINK_DRIVE  
0  https://drive.google.com/file/d/1XJtKtruQ-JQif...  
1  https://drive.google.com/file/d/14FRThExjZmg0R...  
2  https://drive.google.com/file/d/1am

## Send info to cases_gold

In [8]:
import os
import json
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ────────────────────────────────────────────────
# Load Supabase
# ────────────────────────────────────────────────
load_dotenv()

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

# ────────────────────────────────────────────────
# Load our 20 selected cases
# ────────────────────────────────────────────────
cases_df = pd.read_csv(
    "/Users/othmanbensouda/Desktop/debt_collection_website/files/20_rejections_with_drive_links.csv"
)

our_cases = cases_df["CASE_NUMBER"].unique().tolist()

# ────────────────────────────────────────────────
# Load plaintiff/defendant dataset
# ────────────────────────────────────────────────
df = pd.read_csv(
    "/Users/othmanbensouda/Desktop/debt_collection_website/files/get_plaintiff_defendant_rejections.csv"
)

# Keep only our 20 cases
df = df[df["CASE_NUMBER"].isin(our_cases)]

# Convert complaint filed date to date
df["COMPLAINT_FILED_DATE"] = pd.to_datetime(
    df["COMPLAINT_FILED_DATE"],
    errors="coerce"
).dt.date

records = []
allocation_trace = []

annotators = ["Brian", "Victor", "Parker"]

# ────────────────────────────────────────────────
# Build records
# ────────────────────────────────────────────────
for i, (case_number, group) in enumerate(df.groupby("CASE_NUMBER")):

    case_hashkey = str(group["CASE_HASHKEY"].iloc[0])
    complaint_date = group["COMPLAINT_FILED_DATE"].iloc[0]

    plaintiffs = group[group["PERSON_ROLE"] == "Plaintiff"]["FULL_NAME"].tolist()
    defendants = group[group["PERSON_ROLE"] == "Defendant"]["FULL_NAME"].tolist()

    annotator = annotators[i % 3]

    record = {
        "case_number": str(case_number),
        "case_hashkey": case_hashkey,
        "complaint_filed_date": complaint_date.isoformat() if complaint_date else None,
        "plaintiff": plaintiffs,
        "defendant": defendants,
        "annotator_id": annotator,
        "progress": "not_started",
        "round": 1,
        "batch": 3
    }

    records.append(record)

    allocation_trace.append({
        "case_number": str(case_number),
        "annotator_id": annotator,
        "round": 1,
        "batch": 3
    })

# ────────────────────────────────────────────────
# Convert numpy types to safe JSON
# ────────────────────────────────────────────────
records = json.loads(json.dumps(records, default=str))

print(f"Inserting {len(records)} cases into cases_gold...")

supabase.table("cases_gold").insert(records).execute()

print("✅ cases_gold populated successfully.")

# ────────────────────────────────────────────────
# Save allocation trace CSV
# ────────────────────────────────────────────────
allocation_df = pd.DataFrame(allocation_trace)

allocation_path = "/Users/othmanbensouda/Desktop/debt_collection_website/files/cases_gold_allocation_round1_batch3.csv"
allocation_df.to_csv(allocation_path, index=False)

print(f"📄 Allocation trace saved to {allocation_path}")

print("\nAllocation summary:")
print(allocation_df["annotator_id"].value_counts())

Inserting 20 cases into cases_gold...
✅ cases_gold populated successfully.
📄 Allocation trace saved to /Users/othmanbensouda/Desktop/debt_collection_website/files/cases_gold_allocation_round1_batch3.csv

Allocation summary:
annotator_id
Brian     7
Victor    7
Parker    6
Name: count, dtype: int64
